# Experiment 4: Threshold

In this experiment, we will use the best model (dual embeddings of the chunks and the metadata of the whole document) and find the optimal threshold for cosine similarity.

In [ ]:
import os
import json
import time
import requests
from pathlib import Path
import fitz #pip install PyMuPDF

from embedder import Embedder

In [ ]:
BASE_DIR = Path.cwd().parents[0]   # backend/
PDF_EXAMPLES = BASE_DIR/ "example_pdfs_to_upload"

DATA_DIR = BASE_DIR / "data"
META_CACHE_PATH = DATA_DIR / "llm_metadata_cache.json"
LLM_CHUNKS_PATH = BASE_DIR / "llm_database.json" #ATATA CAMBIAR A DATA??

if not META_CACHE_PATH.exists():
    raise RuntimeError("llm_metadata_cache.json not found. Metadata must be precomputed.")

with META_CACHE_PATH.open("r", encoding="utf-8") as f:
    metadata_cache = json.load(f)

In [ ]:
def build_metadata_string(llm_meta):

    if not llm_meta:
        return ""
    parts = []
    if llm_meta.get("title"):
        parts.append(f"Title: {llm_meta['title']}")
    if llm_meta.get("topics"):
        parts.append("Topics: " + ", ".join(llm_meta["topics"]))
    if llm_meta.get("keywords"):
        parts.append("Keywords: " + ", ".join(llm_meta["keywords"]))
    if llm_meta.get("one_sentence_summary"):
        parts.append("Summary: " + llm_meta["one_sentence_summary"])
    return "[METADATA]\n" + "\n".join(parts)

def get_llm_metadata(pdf_name: str) -> dict:
    """
    Read-only metadata accessor.
    No LLM calls. No side effects.
    """
    return metadata_cache.get(pdf_name, {})

In [ ]:
class DataManager:
    """
    Handles:
    - PDF text extraction
    - Chunking
    - Embedding dual
    """

    def __init__(self, pdf_folder, llm_db_path = LLM_CHUNKS_PATH):
        self.embedder = Embedder()
        self.pdf_folder = pdf_folder
        self.tokenizer = self.embedder.model.tokenizer
        
        with open(llm_db_path, "r", encoding="utf-8") as f:
            llm_db = json.load(f)  

        self.llm_chunks_map = {
            d["pdf_name"]: d.get("chunks", [])  
            for d in llm_db
            if "pdf_name" in d
}
    
    # ------------------------------
    # PDF TEXT EXTRACTION
    # ------------------------------
    def extract_text(self, pdf_path):
        doc = fitz.open(pdf_path)
        text = "".join(page.get_text() for page in doc)
        doc.close()
        return text

    # ------------------------------
    # CHUNKING
    # ------------------------------
    
    def get_llm_chunks(self, pdf_name):
        chunks = self.llm_chunks_map.get(pdf_name, [])
        if not chunks:
            raise ValueError(f"No LLM chunks found for {pdf_name} in llm_database.json")
        return chunks

    # ------------------------------
    # MAIN PIPELINE
    # ------------------------------
    def process_pdf(self, pdf_path):
        chunks_from_json = self.get_llm_chunks(pdf_path.name)
        chunks = [c['text'] for c in chunks_from_json]

        llm_meta = get_llm_metadata(pdf_path.name)
        meta_str = build_metadata_string(llm_meta)
        emb_meta = self.embedder.encode(meta_str)

        chunk_records = []
        for chunk in chunks:
            emb_text = self.embedder.encode(chunk)
            chunk_records.append({
                "text": chunk,
                "embedding_text": emb_text,
                "embedding_meta": emb_meta
            })

        return {
            "pdf_name": pdf_path.name,
            "llm_metadata": llm_meta,
            "chunks": chunk_records}

    # ------------------------------
    # INDEX BUILDING
    # ------------------------------
    def build_index(self, pdf_paths):
        database = []

        for pdf_path in pdf_paths:
            entry = self.process_pdf(pdf_path)
            print(f"Indexed: {pdf_path.name}")
            database.append(entry)

        return database

In [ ]:
import numpy as np

class BaselineRetriever:
    def __init__(self, database=None, embedder=None):
        self.embedder = embedder or Embedder()
        self.database = database or []

    def cosine_similarity(self, v1, v2):
        v1 = np.array(v1)
        v2 = np.array(v2)
        return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

    def search(self, query, threshold=0.70, alpha=0.7):
        query_emb = self.embedder.encode(query)
        results = []

        for entry in self.database:
            best_score = 0.0
            best_chunk = None

            for chunk in entry.get("chunks", []):
                s_text = self.cosine_similarity(query_emb, chunk["embedding_text"])
                s_meta = self.cosine_similarity(query_emb, chunk["embedding_meta"])
                score = alpha * s_text + (1 - alpha) * s_meta

                if score > best_score:
                    best_score = score
                    best_chunk = chunk

            if best_score >= threshold:
                results.append({
                    "paper_id": entry.get("pdf_name", ""),
                    "title": entry.get("llm_metadata", {}).get("title", entry.get("pdf_name", "")),
                    "pdf_name": entry.get("pdf_name", ""),
                    "score": round(best_score, 3),
                    "sample_text": best_chunk["text"][:300] if best_chunk else ""
                })

        results.sort(key=lambda x: x["score"], reverse=True)
        return results

In [ ]:
embedder = Embedder()
dm = DataManager(pdf_folder=PDF_EXAMPLES)
pdf_paths = list(PDF_EXAMPLES.glob("*.pdf"))

final_model_db = dm.build_index(pdf_paths)
baseline_retriever = BaselineRetriever(final_model_db, embedder)

### 2. Evaluation

In [ ]:
def hit_at_k_vs_threshold(retrieval_cache, evaluation_pairs, thresholds, k=3):
    hits_by_retriever = {}

    for name, results_by_query in retrieval_cache.items():
        hits_by_retriever[name] = []

        for t in thresholds:
            hits = 0
            for query, expected_pdf in evaluation_pairs.items():
                results = [
                    r for r in results_by_query[query]
                    if r["score"] >= t
                ]

                for r in results[:k]:
                    if r["pdf_name"] == expected_pdf:
                        hits += 1
                        break

            hits_by_retriever[name].append(hits)

    return hits_by_retriever


In [ ]:
thresholds = [1, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0]

hits_by_retriever = hit_at_k_vs_threshold(
    retrieval_easy,
    queries_easy,
    thresholds,
    k=3)

import matplotlib.pyplot as plt

plt.figure()
for name, hits in hits_by_retriever.items():
    plt.plot(thresholds, hits, label=name)

plt.xlabel("Similarity threshold")
plt.ylabel("Number of correct retrieves (Hit@3)")
plt.title("Hit@3 vs Similarity Threshold")
plt.legend()
plt.grid(True)
plt.show()